Install Libraries

In [ ]:
!pip install -U transformers

In [4]:
!pip install transformers datasets seqeval scikit-learn jsonlines

Imports

In [5]:
import json
import jsonlines
import numpy as np

from sklearn.model_selection import train_test_split
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer
)

from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

Label Definitions

In [6]:
labels = [
    "O",
    "B-TOPIC","I-TOPIC",
    "B-METHOD","I-METHOD",
    "B-CONCEPT","I-CONCEPT",
    "B-ALGORITHM","I-ALGORITHM",
    "B-OTHER","I-OTHER"
]

label2id = {l:i for i,l in enumerate(labels)}
id2label = {i:l for l,i in label2id.items()}

Load Dataset

In [7]:
data = []

with jsonlines.open("cleaned_file.jsonl") as reader:
    for obj in reader:
        data.append(obj)

Convert Entities → BIO Labels

In [8]:
def convert_to_tokens(sample):

    text = sample["input"]
    entities = json.loads(sample["output"])

    tokens = text.split()
    tags = ["O"] * len(tokens)

    for ent in entities:

        ent_text = ent["entity"]
        label = ent["label"]

        ent_tokens = ent_text.split()

        for i in range(len(tokens)):

            if tokens[i:i+len(ent_tokens)] == ent_tokens:

                tags[i] = "B-" + label

                for j in range(1,len(ent_tokens)):
                    tags[i+j] = "I-" + label

    return {"tokens":tokens,"ner_tags":tags}

Prepare Dataset

In [9]:
processed = []

for sample in data:
    processed.append(convert_to_tokens(sample))

dataset = Dataset.from_list(processed)

Train/Test Split (90/10)

In [10]:
train_test = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = train_test["train"]
test_dataset = train_test["test"]

Load SciBERT

In [16]:
model_name = "allenai/scibert_scivocab_cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: allenai/scibert_scivocab_cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored wh

Tokenization

In [31]:
MAX_LEN = 512

def tokenize(example):
    tokenized = tokenizer(
        example["tokens"],
        truncation=True,
        max_length=MAX_LEN,        # ✅ truncate sequences to 512
        is_split_into_words=True,
    )

    word_ids = tokenized.word_ids()
    labels_ids = []

    for word_id in word_ids:
        if word_id is None:
            labels_ids.append(-100)
        else:
            label = example["ner_tags"][word_id].upper()
            labels_ids.append(label2id.get(label, label2id["O"]))

    tokenized["labels"] = labels_ids
    return tokenized

In [32]:
train_dataset = train_dataset.map(tokenize)
test_dataset = test_dataset.map(tokenize)

Map:   0%|          | 0/1368 [00:00<?, ? examples/s]

Map:   0%|          | 0/152 [00:00<?, ? examples/s]

Training Arguments

In [33]:
training_args = TrainingArguments(

    output_dir="scibert_ner",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=3,

    weight_decay=0.01,

    eval_strategy="epoch",

    save_strategy="epoch",

    logging_dir="./logs"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Metrics Function

In [34]:
def compute_metrics(p):

    predictions, labels = p

    predictions = np.argmax(predictions, axis=2)

    true_labels = []
    true_predictions = []

    for pred, lab in zip(predictions, labels):

        temp_pred = []
        temp_lab = []

        for p_i,l_i in zip(pred,lab):

            if l_i != -100:
                temp_pred.append(id2label[p_i])
                temp_lab.append(id2label[l_i])

        true_predictions.append(temp_pred)
        true_labels.append(temp_lab)

    return {

        "precision": precision_score(true_labels,true_predictions),
        "recall": recall_score(true_labels,true_predictions),
        "f1": f1_score(true_labels,true_predictions)
    }

Trainer

In [35]:
from transformers import DataCollatorForTokenClassification

# This will pad both input_ids and labels to the longest sequence in the batch
data_collator = DataCollatorForTokenClassification(tokenizer, padding=True)

# Trainer
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,  # ✅ ensures consistent batch lengths
    compute_metrics=compute_metrics
)

Train Model

In [36]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,No log,0.037428,0.870699,0.935050,0.901728
2,No log,0.030767,0.870314,0.961926,0.913830
3,0.034271,0.028745,0.895397,0.958567,0.925906


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=513, training_loss=0.03374291622383088, metrics={'train_runtime': 197.6037, 'train_samples_per_second': 20.769, 'train_steps_per_second': 2.596, 'total_flos': 424404959874144.0, 'train_loss': 0.03374291622383088, 'epoch': 3.0})

Evaluate Model

In [39]:
import numpy as np
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report
from sklearn.metrics import accuracy_score

# Get predictions (this works regardless of evaluate hooks)
predictions, labels, _ = trainer.predict(test_dataset)
predictions = np.argmax(predictions, axis=2)

true_labels = []
true_predictions = []

for pred_seq, label_seq in zip(predictions, labels):
    temp_pred, temp_lab = [], []
    for p_i, l_i in zip(pred_seq, label_seq):
        if l_i != -100:  # ignore padding
            temp_pred.append(id2label[p_i])
            temp_lab.append(id2label[l_i])
    true_predictions.append(temp_pred)
    true_labels.append(temp_lab)

# Compute metrics
micro_precision = precision_score(true_labels, true_predictions)
micro_recall = recall_score(true_labels, true_predictions)
micro_f1 = f1_score(true_labels, true_predictions)

report = classification_report(true_labels, true_predictions, output_dict=True)
macro_f1 = report["macro avg"]["f1-score"]

# Exact accuracy
flat_true, flat_pred = [], []
for t_seq, p_seq in zip(true_labels, true_predictions):
    flat_true.extend(t_seq)
    flat_pred.extend(p_seq)
exact_accuracy = accuracy_score(flat_true, flat_pred)

# Partial accuracy
partial_matches = 0
total_tokens = 0
for t_seq, p_seq in zip(true_labels, true_predictions):
    for t, p in zip(t_seq, p_seq):
        total_tokens += 1
        if t == p:
            partial_matches += 1
        elif t != "O" and p != "O" and t.split("-")[-1] == p.split("-")[-1]:
            partial_matches += 0.5
partial_accuracy = partial_matches / total_tokens

# Print metrics
print("\n===== Evaluation Metrics =====\n")
print("Micro Precision :", round(micro_precision,4))
print("Micro Recall    :", round(micro_recall,4))
print("Micro F1        :", round(micro_f1,4))
print("Macro F1        :", round(macro_f1,4))
print("Exact Accuracy  :", round(exact_accuracy,4))
print("Partial Accuracy:", round(partial_accuracy,4))
print("\nDetailed Classification Report:\n")
print(classification_report(true_labels, true_predictions))


===== Evaluation Metrics =====

Micro Precision : 0.8954
Micro Recall    : 0.9586
Micro F1        : 0.9259
Macro F1        : 0.9019
Exact Accuracy  : 0.9911
Partial Accuracy: 0.9912

Detailed Classification Report:

              precision    recall  f1-score   support

     CONCEPT       0.92      0.95      0.93       302
      METHOD       0.81      0.87      0.84        15
       OTHER       0.90      0.96      0.93       462
       TOPIC       0.84      0.98      0.91       114

   micro avg       0.90      0.96      0.93       893
   macro avg       0.87      0.94      0.90       893
weighted avg       0.90      0.96      0.93       893

